In [5]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.linear_model import ElasticNet

warnings.filterwarnings("ignore")

# ====================== 1. 读取数据 ======================
# ============== 1. 读取数据 ==============
df = pd.read_excel(r"D:\文成数据库\Nb-Si数据库2.28-高温压缩.xlsx")

# ============== 2. 定义特征列表 & 目标列 ==============
features_name = [
    "Nb", "Si", "Ti", "Al", "Cr", "Hf", "Zr", "Mo", "Fe", "B", "V", 
    "VEC", "x", "Δx", "ΔHmix", "ΔSmix", "ΔG", "Tm", "ΔTm", "ρ", "r", "am", "Δa", "δ", "Ω", "λ",
    "Mean_A1 atomic number",
    "Mean_A6 atomic weight",
    "Mean_E2 electronegativity (Pauling)",
    "Mean_E5 energy ionization first",
    "Mean_E13 Electron affinity",
    "Mean_Electrophilicity Index",
    "Mean_Speed of Sound",
    "Mean_Neutron Cross Section",
    "Mean_Neutron Mass Absorption",
    "Mean_Space Group Number",
    "Mean_Glawe Number",
    "Mean_C1 temperature melting",
    "Mean_C2 temperature boiling",
    "Mean_density",
    "Mean_C3 enthalpy vaporization",
    "Mean_C4 enthalpy melting",
    "Mean_C5 enthalpy atomization",
    "Mean_热导率W/(mk)",
    "Mean_电导率(MS/m)",
    "Mean_电阻率(mΩ)",
    "Mean_热膨胀(1/k)",
    "Mean_比热容J/g/k",
    "Mean_摩尔热容(J/mol/k)",
    "Mean_摩尔体积(cm3/mol)",
    "Mean_莫氏硬度(MPa)",
    "Mean_covalent radius Cordero (pm)",
    "Mean_covalent radius Pyykko(Single Bond) (pm)",
    "Mean_covalent radius Pyykko(Double Bond) (pm)",
    "Mean_Pyykko(Triple Bond) (pm)",
    "Mean_S10 Lattice Constants a",
    "Mean_S11 Lattice Constants b",
    "Mean_S12 Lattice Constants c",
    "Mean_S13 radii atomic (coordination number 12) (pm)",
    "Mean_metallic radius(pm)",
    "Mean_metallic radius 12 Neighbors(pm)",
    
    "Var_A1 atomic number",
    "Var_A6 atomic weight",
    "Var_E2 electronegativity (Pauling)",
    "Var_E5 energy ionization first",
    "Var_E13 Electron affinity",
    "Var_Electrophilicity Index",
    "Var_Speed of Sound",
    "Var_Neutron Cross Section",
    "Var_Neutron Mass Absorption",
    "Var_Space Group Number",
    "Var_Glawe Number",
    "Var_C1 temperature melting",
    "Var_C2 temperature boiling",
    "Var_density",
    "Var_C3 enthalpy vaporization",
    "Var_C4 enthalpy melting",
    "Var_C5 enthalpy atomization",
    "Var_热导率W/(mk)",
    "Var_电导率(MS/m)",
    "Var_电阻率(mΩ)",
    "Var_热膨胀(1/k)",
    "Var_比热容J/g/k",
    "Var_摩尔热容(J/mol/k)",
    "Var_摩尔体积(cm3/mol)",
    "Var_莫氏硬度(MPa)",
    "Var_covalent radius Cordero (pm)",
    "Var_covalent radius Pyykko(Single Bond) (pm)",
    "Var_covalent radius Pyykko(Double Bond) (pm)",
    "Var_Pyykko(Triple Bond) (pm)",
    "Var_S10 Lattice Constants a",
    "Var_S11 Lattice Constants b",
    "Var_S12 Lattice Constants c",
    "Var_S13 radii atomic (coordination number 12) (pm)",
    "Var_metallic radius(pm)",
    "Var_metallic radius 12 Neighbors(pm)",
]
target_col = 'high temperature compression'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# -------------- 提取X与y --------------
X_all = df[features_name].values
y_all = df[target_col].values
y_all_np = np.array(y_all, dtype=float)

# ==== 可视化结果保存路径 ====
viz_save_dir = r"D:\文成数据库\6.12-htcR2 rmse mae可视化"
if not os.path.exists(viz_save_dir):
    os.makedirs(viz_save_dir)

# ====================== 2. 单次标准化 (修改后) ======================
# 只对原始数据标准化一次，用于特征选择
scaler_for_selection = StandardScaler()
X_all_scaled = scaler_for_selection.fit_transform(X_all)

n_samples, n_features = X_all_scaled.shape

# ====================== 3. PCC 去除冗余(特征-特征) ======================
pcc_matrix = np.zeros((n_features, n_features))
for i in range(n_features):
    for j in range(i + 1, n_features):
        pcc_val, _ = pearsonr(X_all_scaled[:, i], X_all_scaled[:, j])
        pcc_matrix[i, j] = pcc_val
        pcc_matrix[j, i] = pcc_val

pcc_threshold = 0.95
redundant_features = set()
redundancy_logs = []

print("### PCC 去除冗余特征过程：")
print("| 冗余特征对 | 特征对 PCC 值 | 移除特征 | 移除特征与目标变量 PCC 值 |")
print("| --- | --- | --- | --- |")

for i in range(n_features):
    for j in range(i + 1, n_features):
        if abs(pcc_matrix[i, j]) > pcc_threshold:
            # 比较两个特征与目标变量的相关性
            pcc_i, _ = pearsonr(X_all_scaled[:, i], y_all_np)
            pcc_j, _ = pearsonr(X_all_scaled[:, j], y_all_np)
            if abs(pcc_i) < abs(pcc_j):
                redundant_features.add(i)
                removed_feature = features_name[i]
                removed_pcc = pcc_i
            else:
                redundant_features.add(j)
                removed_feature = features_name[j]
                removed_pcc = pcc_j
            log = {
                "冗余特征对": f"{features_name[i]} 和 {features_name[j]}",
                "特征对 PCC 值": pcc_matrix[i, j],
                "移除特征": removed_feature,
                "移除特征与目标变量 PCC 值": removed_pcc
            }
            redundancy_logs.append(log)
            print(f"| {features_name[i]} 和 {features_name[j]} | {pcc_matrix[i, j]:.4f} | {removed_feature} | {removed_pcc:.4f} |")

non_redundant_indices = [i for i in range(n_features) if i not in redundant_features]
X_nonredundant = X_all_scaled[:, non_redundant_indices]
nr_features_name = [features_name[i] for i in non_redundant_indices]

# ====================== 4. 过滤法：PCC + MIC (与目标) ======================
#  4.1 计算与目标的Pearson相关(绝对值大于一定阈值)
corr_threshold = 0.09
pcc_with_target = []
for i in range(X_nonredundant.shape[1]):
    p_val, _ = pearsonr(X_nonredundant[:, i], y_all_np)
    pcc_with_target.append(abs(p_val))

pcc_indices = [i for i, v in enumerate(pcc_with_target) if v > corr_threshold]

#  4.2 计算与目标的互信息(MIC)
mic_values = mutual_info_regression(X_nonredundant, y_all_np)
mic_threshold = 0.08
mic_indices = [i for i, v in enumerate(mic_values) if v > mic_threshold]

#  4.3 合并策略(交集)
selected_indices_filter = set(pcc_indices).intersection(set(mic_indices))
selected_indices_filter = sorted(list(selected_indices_filter))

filtered_features = [nr_features_name[i] for i in selected_indices_filter]
X_filter = X_nonredundant[:, selected_indices_filter]

print("\n=== 过滤法(PCC+MIC)之后保留的特征 ===")
print(filtered_features)

# ====================== 5. 包装法(RFE + ElasticNet) ======================
#  以ElasticNet作为基模型进行RFE
n_features_to_select = 6
elasticnet_estimator = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=0)
selector = RFE(estimator=elasticnet_estimator, n_features_to_select=n_features_to_select)
selector = selector.fit(X_filter, y_all_np)

selected_mask_rfe = selector.support_
final_features_rfe = [filtered_features[i] for i, sel in enumerate(selected_mask_rfe) if sel]

print("\n=== RFE(ElasticNet)后保留的最终6个特征 ===")
print(final_features_rfe)

# ====================== 6. 提取最终特征并标准化 (修改后) ======================
# 从原始数据中提取最终选择的特征
X_final_original = df[final_features_rfe].values

# 对最终特征进行标准化(这是唯一需要保存的标准化器)
final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_final_original)

# 为神经网络构建特征和目标数据
feature_rfe = pd.DataFrame(X_final_scaled, columns=final_features_rfe)
target = pd.Series(y_all_np, name=target_col)

# ====================== 7. 定义模型 ======================
class MyModel(nn.Module):
    def __init__(self, input_dim=12):
        super(MyModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 450)
        self.bn1 = nn.BatchNorm1d(450)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(450, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

my_nn = MyModel(len(final_features_rfe)).to(device)

# ====================== 8. 定义损失函数与优化器 ======================
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(my_nn.parameters(), lr=0.01, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

# ====================== 9. 日志记录相关变量 ======================
train_loss_history = []
test_loss_history = []
train_r2_history = []
test_r2_history = []
train_rmse_history = []
test_rmse_history = []
# 新增MAE指标记录
train_mae_history = []
test_mae_history = []

# 新增MAPE, MRE, MRE+SD指标记录
train_mape_history = []
test_mape_history = []
train_mre_history = []
test_mre_history = []
train_mre_sd_history = []
test_mre_sd_history = []

rolling_train_r2 = []
rolling_test_r2 = []
rolling_train_rmse = []
rolling_test_rmse = []
rolling_train_mae = []
rolling_test_mae = []

# 新增滚动MAPE, MRE, MRE+SD记录
rolling_train_mape = []
rolling_test_mape = []
rolling_train_mre = []
rolling_test_mre = []
rolling_train_mre_sd = []
rolling_test_mre_sd = []

rolling_train_r21 = []
rolling_test_r21 = []
rolling_train_rmse1 = []
rolling_test_rmse1 = []
rolling_train_mae1 = []
rolling_test_mae1 = []

# 新增详细MAPE, MRE, MRE+SD记录
rolling_train_mape1 = []
rolling_test_mape1 = []
rolling_train_mre1 = []
rolling_test_mre1 = []
rolling_train_mre_sd1 = []
rolling_test_mre_sd1 = []

avg_train_r2 = None
avg_test_r2 = None
avg_train_rmse = None
avg_test_rmse = None
avg_train_mae = None
avg_test_mae = None

# 新增平均MAPE, MRE, MRE+SD记录
avg_train_mape = None
avg_test_mape = None
avg_train_mre = None
avg_test_mre = None
avg_train_mre_sd = None
avg_test_mre_sd = None

output_step = 0

# 用于存储最终模型的预测结果
final_train_actual = None
final_train_pred = None
final_test_actual = None
final_test_pred = None
final_train_features = None
final_test_features = None
final_best_seed = None

# ====================== 新增：定义MAPE, MRE, MRE+SD计算函数 ======================
def calculate_mape(y_true, y_pred):
    """计算平均绝对百分比误差 (Mean Absolute Percentage Error)"""
    # 避免除以零
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def calculate_mre(y_true, y_pred):
    """计算平均相对误差 (Mean Relative Error)"""
    # 避免除以零
    mask = y_true != 0
    return np.mean((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100

def calculate_mre_sd(y_true, y_pred):
    """计算MRE+SD (平均相对误差+标准差)"""
    # 避免除以零
    mask = y_true != 0
    mre = np.mean((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100
    
    # 计算相对误差的标准差
    re_values = ((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100
    sd = np.std(re_values)
    
    return mre, sd, mre + sd

# ====================== 10. 训练循环 ======================
for epoch in range(1000000):
    # 每轮随机选一个种子来拆分数据
    i = random.randint(0, 4294967295)

    x_train_np, x_test_np, y_train_np, y_test_np = train_test_split(
        feature_rfe.values, target.values, test_size=0.2, random_state=i
    )

    train_x = torch.from_numpy(x_train_np.astype(np.float32)).to(device)
    train_y = torch.from_numpy(y_train_np.astype(np.float32)).to(device).view(-1, 1)
    test_x = torch.from_numpy(x_test_np.astype(np.float32)).to(device)
    test_y = torch.from_numpy(y_test_np.astype(np.float32)).to(device).view(-1, 1)

    # ---------- 前向传播、反向传播 ----------
    my_nn.train()
    optimizer.zero_grad()
    outputs = my_nn(train_x)
    loss = criterion(outputs, train_y)
    loss.backward()
    optimizer.step()
    train_loss = loss.item()

    # 更新学习率
    scheduler.step()

    # ---------- 评估 ----------
    my_nn.eval()
    with torch.no_grad():
        outputs_train = my_nn(train_x)
        r2_train = r2_score(y_train_np, outputs_train.cpu().numpy())
        rmse_train = np.sqrt(mean_squared_error(y_train_np, outputs_train.cpu().numpy()))
        mae_train = mean_absolute_error(y_train_np, outputs_train.cpu().numpy())
        
        # 计算新指标
        mape_train = calculate_mape(y_train_np, outputs_train.cpu().numpy().flatten())
        mre_train = calculate_mre(y_train_np, outputs_train.cpu().numpy().flatten())
        mre_train_val, mre_sd_train_val, mre_sd_train = calculate_mre_sd(y_train_np, outputs_train.cpu().numpy().flatten())

        outputs_test = my_nn(test_x)
        r2_test = r2_score(y_test_np, outputs_test.cpu().numpy())
        rmse_test = np.sqrt(mean_squared_error(y_test_np, outputs_test.cpu().numpy()))
        mae_test = mean_absolute_error(y_test_np, outputs_test.cpu().numpy())
        
        # 计算新指标
        mape_test = calculate_mape(y_test_np, outputs_test.cpu().numpy().flatten())
        mre_test = calculate_mre(y_test_np, outputs_test.cpu().numpy().flatten())
        mre_test_val, mre_sd_test_val, mre_sd_test = calculate_mre_sd(y_test_np, outputs_test.cpu().numpy().flatten())
        
        test_loss = criterion(outputs_test, test_y).item()

    # 记录Loss，用于可视化
    train_loss_history.append(train_loss)
    test_loss_history.append(test_loss)

    # ---------- 滚动平均 ----------
    rolling_train_r2.append(r2_train)
    rolling_test_r2.append(r2_test)
    rolling_train_rmse.append(rmse_train)
    rolling_test_rmse.append(rmse_test)
    rolling_train_mae.append(mae_train)
    rolling_test_mae.append(mae_test)
    
    # 新指标滚动记录
    rolling_train_mape.append(mape_train)
    rolling_test_mape.append(mape_test)
    rolling_train_mre.append(mre_train)
    rolling_test_mre.append(mre_test)
    rolling_train_mre_sd.append(mre_sd_train)
    rolling_test_mre_sd.append(mre_sd_test)

    rolling_train_r21.append(r2_train)
    rolling_test_r21.append(r2_test)
    rolling_train_rmse1.append(rmse_train)
    rolling_test_rmse1.append(rmse_test)
    rolling_train_mae1.append(mae_train)
    rolling_test_mae1.append(mae_test)
    
    # 新指标详细记录
    rolling_train_mape1.append(mape_train)
    rolling_test_mape1.append(mape_test)
    rolling_train_mre1.append(mre_train)
    rolling_test_mre1.append(mre_test)
    rolling_train_mre_sd1.append(mre_sd_train)
    rolling_test_mre_sd1.append(mre_sd_test)

    # 每 5 次迭代计算一次平均
    if len(rolling_train_r2) == 5:
        avg_train_r2 = np.mean(rolling_train_r2)
        avg_test_r2 = np.mean(rolling_test_r2)
        avg_train_rmse = np.mean(rolling_train_rmse)
        avg_test_rmse = np.mean(rolling_test_rmse)
        avg_train_mae = np.mean(rolling_train_mae)
        avg_test_mae = np.mean(rolling_test_mae)
        
        # 新指标平均计算
        avg_train_mape = np.mean(rolling_train_mape)
        avg_test_mape = np.mean(rolling_test_mape)
        avg_train_mre = np.mean(rolling_train_mre)
        avg_test_mre = np.mean(rolling_test_mre)
        avg_train_mre_sd = np.mean(rolling_train_mre_sd)
        avg_test_mre_sd = np.mean(rolling_test_mre_sd)

        train_r2_history.append(avg_train_r2)
        test_r2_history.append(avg_test_r2)
        train_rmse_history.append(avg_train_rmse)
        test_rmse_history.append(avg_test_rmse)
        train_mae_history.append(avg_train_mae)
        test_mae_history.append(avg_test_mae)
        
        # 新指标历史记录
        train_mape_history.append(avg_train_mape)
        test_mape_history.append(avg_test_mape)
        train_mre_history.append(avg_train_mre)
        test_mre_history.append(avg_test_mre)
        train_mre_sd_history.append(avg_train_mre_sd)
        test_mre_sd_history.append(avg_test_mre_sd)

        # 清空滚动
        rolling_train_r2 = []
        rolling_test_r2 = []
        rolling_train_rmse = []
        rolling_test_rmse = []
        rolling_train_mae = []
        rolling_test_mae = []
        
        # 新指标滚动清空
        rolling_train_mape = []
        rolling_test_mape = []
        rolling_train_mre = []
        rolling_test_mre = []
        rolling_train_mre_sd = []
        rolling_test_mre_sd = []

        output_step += 1

        # 修改：每1000次训练打印一次（即每200个output_step）
        if output_step == 200:
            print(
                f"次数: {epoch} | 训练集 R2: {avg_train_r2:.4f} | 训练集 RMSE: {avg_train_rmse:.4f} | 训练集 MAE: {avg_train_mae:.4f} "
                f"| 测试集 R2: {avg_test_r2:.4f} | 测试集 RMSE: {avg_test_rmse:.4f} | 测试集 MAE: {avg_test_mae:.4f} "
                f"| LR: {optimizer.param_groups[0]['lr']}"
            )

            # 当达到保存条件时，训练结束
            if ((avg_train_r2 > 0.91 and avg_test_r2 > 0.91)
                and (avg_train_rmse < 7 and avg_test_rmse < 7)
            ):
                # 保存当前的预测结果，用于后续散点图可视化
                final_train_actual = y_train_np.copy()
                final_train_pred = outputs_train.cpu().numpy().flatten()
                final_test_actual = y_test_np.copy()
                final_test_pred = outputs_test.cpu().numpy().flatten()
                final_train_features = x_train_np.copy()
                final_test_features = x_test_np.copy()
                final_best_seed = i
                
                print(f"模型在第 {epoch} 轮训练后达到条件！| 随机种子： {i}")
                break  # 终止训练

            output_step = 0

else:
    print("训练结束，未达到模型保存条件。")
    if train_r2_history and test_r2_history and train_rmse_history and test_rmse_history:
        final_avg_train_r2 = train_r2_history[-1]
        final_avg_test_r2 = test_r2_history[-1]
        final_avg_train_rmse = train_rmse_history[-1]
        final_avg_test_rmse = test_rmse_history[-1]
        final_avg_train_mae = train_mae_history[-1]
        final_avg_test_mae = test_mae_history[-1]
        final_avg_train_mape = train_mape_history[-1]
        final_avg_test_mape = test_mape_history[-1]
        final_avg_train_mre = train_mre_history[-1]
        final_avg_test_mre = test_mre_history[-1]
        final_avg_train_mre_sd = train_mre_sd_history[-1]
        final_avg_test_mre_sd = test_mre_sd_history[-1]
        
        print(f"最终训练集 R2: {final_avg_train_r2:.4f} | 最终训练集 RMSE: {final_avg_train_rmse:.4f} | 最终训练集 MAE: {final_avg_train_mae:.4f}")
        print(f"最终测试集 R2: {final_avg_test_r2:.4f} | 最终测试集 RMSE: {final_avg_test_rmse:.4f} | 最终测试集 MAE: {final_avg_test_mae:.4f}")
        print(f"最终训练集 MAPE: {final_avg_train_mape:.4f} | 最终训练集 MRE: {final_avg_train_mre:.4f} | 最终训练集 MRE+SD: {final_avg_train_mre_sd:.4f}")
        print(f"最终测试集 MAPE: {final_avg_test_mape:.4f} | 最终测试集 MRE: {final_avg_test_mre:.4f} | 最终测试集 MRE+SD: {final_avg_test_mre_sd:.4f}")

    # 如果没有达到终止条件，我们仍然需要保存最后一轮的预测结果用于散点图
    with torch.no_grad():
        final_train_actual = y_train_np.copy()
        final_train_pred = outputs_train.cpu().numpy().flatten()
        final_test_actual = y_test_np.copy()
        final_test_pred = outputs_test.cpu().numpy().flatten()
        final_train_features = x_train_np.copy()
        final_test_features = x_test_np.copy()
        final_best_seed = i

# ====================== 11. 可视化 ======================
# 1. Loss 曲线可视化
plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
plt.plot(train_loss_history, label='Train Loss')
plt.plot(test_loss_history, label='Test Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Train / Test Loss Curve')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(train_r2_history, label='Train R²')
plt.plot(test_r2_history, label='Test R²')
plt.xlabel('50 Iterations')
plt.ylabel('R²')
plt.title('Train / Test R² Curve')
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(viz_save_dir, 'loss_and_r2_curves.png'), dpi=300)
plt.close()

# 2. R² 可视化
plt.figure(figsize=(10, 6))
plt.plot(train_r2_history, label='Train R²', color='blue')
plt.plot(test_r2_history, label='Test R²', color='red')
plt.xlabel('50 Iterations')
plt.ylabel('R²')
plt.title('R² Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'r2_evolution.png'), dpi=300)
plt.close()

# 3. RMSE 可视化
plt.figure(figsize=(10, 6))
plt.plot(train_rmse_history, label='Train RMSE', color='blue')
plt.plot(test_rmse_history, label='Test RMSE', color='red')
plt.xlabel('50 Iterations')
plt.ylabel('RMSE')
plt.title('RMSE Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'rmse_evolution.png'), dpi=300)
plt.close()

# 4. MAE 可视化
plt.figure(figsize=(10, 6))
plt.plot(train_mae_history, label='Train MAE', color='blue')
plt.plot(test_mae_history, label='Test MAE', color='red')
plt.xlabel('50 Iterations')
plt.ylabel('MAE')
plt.title('MAE Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'mae_evolution.png'), dpi=300)
plt.close()

# 5. MAPE 可视化
plt.figure(figsize=(10, 6))
plt.plot(train_mape_history, label='Train MAPE', color='blue')
plt.plot(test_mape_history, label='Test MAPE', color='red')
plt.xlabel('50 Iterations')
plt.ylabel('MAPE (%)')
plt.title('MAPE Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'mape_evolution.png'), dpi=300)
plt.close()

# 6. MRE 可视化
plt.figure(figsize=(10, 6))
plt.plot(train_mre_history, label='Train MRE', color='blue')
plt.plot(test_mre_history, label='Test MRE', color='red')
plt.xlabel('50 Iterations')
plt.ylabel('MRE (%)')
plt.title('MRE Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'mre_evolution.png'), dpi=300)
plt.close()

# 7. MRE+SD 可视化
plt.figure(figsize=(10, 6))
plt.plot(train_mre_sd_history, label='Train MRE+SD', color='blue')
plt.plot(test_mre_sd_history, label='Test MRE+SD', color='red')
plt.xlabel('50 Iterations')
plt.ylabel('MRE+SD (%)')
plt.title('MRE+SD Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'mre_sd_evolution.png'), dpi=300)
plt.close()

# 8. 组合指标可视化 (所有指标)
fig, axes = plt.subplots(6, 1, figsize=(12, 24), sharex=True)

# R²
axes[0].plot(train_r2_history, label='Train R²', color='blue')
axes[0].plot(test_r2_history, label='Test R²', color='red')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Evolution')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.7)

# RMSE
axes[1].plot(train_rmse_history, label='Train RMSE', color='blue')
axes[1].plot(test_rmse_history, label='Test RMSE', color='red')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE Evolution')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.7)

# MAE
axes[2].plot(train_mae_history, label='Train MAE', color='blue')
axes[2].plot(test_mae_history, label='Test MAE', color='red')
axes[2].set_ylabel('MAE')
axes[2].set_title('MAE Evolution')
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.7)

# MAPE
axes[3].plot(train_mape_history, label='Train MAPE', color='blue')
axes[3].plot(test_mape_history, label='Test MAPE', color='red')
axes[3].set_ylabel('MAPE (%)')
axes[3].set_title('MAPE Evolution')
axes[3].legend()
axes[3].grid(True, linestyle='--', alpha=0.7)

# MRE
axes[4].plot(train_mre_history, label='Train MRE', color='blue')
axes[4].plot(test_mre_history, label='Test MRE', color='red')
axes[4].set_ylabel('MRE (%)')
axes[4].set_title('MRE Evolution')
axes[4].legend()
axes[4].grid(True, linestyle='--', alpha=0.7)

# MRE+SD
axes[5].plot(train_mre_sd_history, label='Train MRE+SD', color='blue')
axes[5].plot(test_mre_sd_history, label='Test MRE+SD', color='red')
axes[5].set_xlabel('50 Iterations')
axes[5].set_ylabel('MRE+SD (%)')
axes[5].set_title('MRE+SD Evolution')
axes[5].legend()
axes[5].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig(os.path.join(viz_save_dir, 'all_metrics.png'), dpi=300)
plt.close()

# 9. 详细训练曲线 (所有单次训练的点)
fig, axes = plt.subplots(6, 1, figsize=(12, 24), sharex=True)

# R²
axes[0].plot(rolling_train_r21, label='Train R²', color='blue', alpha=0.3)
axes[0].plot(rolling_test_r21, label='Test R²', color='red', alpha=0.3)
axes[0].set_ylabel('R²')
axes[0].set_title('Detailed R² Evolution (Every Training)')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.7)

# RMSE
axes[1].plot(rolling_train_rmse1, label='Train RMSE', color='blue', alpha=0.3)
axes[1].plot(rolling_test_rmse1, label='Test RMSE', color='red', alpha=0.3)
axes[1].set_ylabel('RMSE')
axes[1].set_title('Detailed RMSE Evolution (Every Training)')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.7)

# MAE
axes[2].plot(rolling_train_mae1, label='Train MAE', color='blue', alpha=0.3)
axes[2].plot(rolling_test_mae1, label='Test MAE', color='red', alpha=0.3)
axes[2].set_ylabel('MAE')
axes[2].set_title('Detailed MAE Evolution (Every Training)')
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.7)

# MAPE
axes[3].plot(rolling_train_mape1, label='Train MAPE', color='blue', alpha=0.3)
axes[3].plot(rolling_test_mape1, label='Test MAPE', color='red', alpha=0.3)
axes[3].set_ylabel('MAPE (%)')
axes[3].set_title('Detailed MAPE Evolution (Every Training)')
axes[3].legend()
axes[3].grid(True, linestyle='--', alpha=0.7)

# MRE
axes[4].plot(rolling_train_mre1, label='Train MRE', color='blue', alpha=0.3)
axes[4].plot(rolling_test_mre1, label='Test MRE', color='red', alpha=0.3)
axes[4].set_ylabel('MRE (%)')
axes[4].set_title('Detailed MRE Evolution (Every Training)')
axes[4].legend()
axes[4].grid(True, linestyle='--', alpha=0.7)

# MRE+SD
axes[5].plot(rolling_train_mre_sd1, label='Train MRE+SD', color='blue', alpha=0.3)
axes[5].plot(rolling_test_mre_sd1, label='Test MRE+SD', color='red', alpha=0.3)
axes[5].set_xlabel('Iteration')
axes[5].set_ylabel('MRE+SD (%)')
axes[5].set_title('Detailed MRE+SD Evolution (Every Training)')
axes[5].legend()
axes[5].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig(os.path.join(viz_save_dir, 'detailed_all_metrics.png'), dpi=300)
plt.close()

# ====================== 12. 散点图可视化 ======================
if final_train_actual is not None and final_train_pred is not None and final_test_actual is not None and final_test_pred is not None:
    # 确定数据范围，以便画对角线
    min_val = min(final_train_actual.min(), final_test_actual.min(), final_train_pred.min(), final_test_pred.min())
    max_val = max(final_train_actual.max(), final_test_actual.max(), final_train_pred.max(), final_test_pred.max())
    
    # 添加一点边距
    margin = (max_val - min_val) * 0.1
    min_val -= margin
    max_val += margin
    
    # 1. 训练集散点图
    plt.figure(figsize=(10, 8))
    plt.scatter(final_train_actual, final_train_pred, alpha=0.7, color='blue', label=f'Train Data (R² = {r2_score(final_train_actual, final_train_pred):.4f})')
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title('Training Set: Actual vs. Predicted Values')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.axis('equal')
    plt.savefig(os.path.join(viz_save_dir, 'train_scatter_plot.png'), dpi=300)
    plt.close()
    
    # 2. 测试集散点图
    plt.figure(figsize=(10, 8))
    plt.scatter(final_test_actual, final_test_pred, alpha=0.7, color='red', label=f'Test Data (R² = {r2_score(final_test_actual, final_test_pred):.4f})')
    plt.plot([min_val, max_val], [min_val, max_val], 'b--', label='Perfect Prediction')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title('Test Set: Actual vs. Predicted Values')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.axis('equal')
    plt.savefig(os.path.join(viz_save_dir, 'test_scatter_plot.png'), dpi=300)
    plt.close()
    
    # 3. 组合散点图
    plt.figure(figsize=(10, 8))
    plt.scatter(final_train_actual, final_train_pred, alpha=0.7, color='blue', label=f'Train (R² = {r2_score(final_train_actual, final_train_pred):.4f})')
    plt.scatter(final_test_actual, final_test_pred, alpha=0.7, color='red', label=f'Test (R² = {r2_score(final_test_actual, final_test_pred):.4f})')
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='Perfect Prediction')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title('Combined: Actual vs. Predicted Values')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.axis('equal')
    plt.savefig(os.path.join(viz_save_dir, 'combined_scatter_plot.png'), dpi=300)
    plt.close()
    
    # 计算残差
    train_residuals = final_train_actual - final_train_pred
    test_residuals = final_test_actual - final_test_pred
    
    # 4. 残差散点图
    plt.figure(figsize=(10, 8))
    plt.scatter(final_train_pred, train_residuals, alpha=0.7, color='blue', label='Train')
    plt.scatter(final_test_pred, test_residuals, alpha=0.7, color='red', label='Test')
    plt.axhline(y=0, color='k', linestyle='--')
    plt.xlabel('Predicted Values')
    plt.ylabel('Residuals (Actual - Predicted)')
    plt.title('Residual Plot')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(viz_save_dir, 'residual_plot.png'), dpi=300)
    plt.close()
    
    # ====================== 13. 计算数据库中每个数据的MAPE, MRE, MRE+SD ======================
    # 使用训练好的模型预测整个数据集
    my_nn.eval()
    with torch.no_grad():
        # 转换所有数据为张量并预测
        all_features_tensor = torch.from_numpy(X_final_scaled.astype(np.float32)).to(device)
        all_predictions = my_nn(all_features_tensor).cpu().numpy().flatten()
    
    # 计算每个样本的相对误差和绝对百分比误差
    individual_abs_percent_errors = np.zeros_like(y_all_np)
    individual_rel_errors = np.zeros_like(y_all_np)
    
    # 新增: 计算每个样本的MAE、MSE、RMSE
    individual_abs_errors = np.abs(y_all_np - all_predictions)  # 绝对误差 (MAE)
    individual_sq_errors = np.square(y_all_np - all_predictions)  # 平方误差 (MSE)
    individual_rmse = np.sqrt(individual_sq_errors)  # 平方根误差 (RMSE)
    
    # 避免除以零
    mask = y_all_np != 0
    individual_abs_percent_errors[mask] = np.abs((y_all_np[mask] - all_predictions[mask]) / y_all_np[mask]) * 100
    individual_rel_errors[mask] = ((all_predictions[mask] - y_all_np[mask]) / y_all_np[mask]) * 100
    
    # 计算总体指标
    overall_mape = np.mean(individual_abs_percent_errors[mask])
    overall_mre = np.mean(individual_rel_errors[mask])
    overall_mre_sd = np.std(individual_rel_errors[mask])
    overall_mre_plus_sd = overall_mre + overall_mre_sd
    
    # 新增: 计算总体MAE、MSE、RMSE指标
    overall_mae = np.mean(individual_abs_errors)
    overall_mse = np.mean(individual_sq_errors)
    overall_rmse = np.sqrt(overall_mse)  # 或者使用: np.mean(individual_rmse)
    
    # ====================== 14. 创建数据表格并保存到Excel ======================
    # 创建训练和测试集数据
    train_df = pd.DataFrame()
    for i, feature in enumerate(final_features_rfe):
        # 使用反标准化获取原始特征值
        train_df[feature] = final_scaler.inverse_transform(final_train_features)[:, i]
    
    train_df['Actual'] = final_train_actual
    train_df['Predicted'] = final_train_pred
    train_df['Residual'] = train_residuals
    train_df['APE (%)'] = np.zeros_like(final_train_actual)  # 绝对百分比误差
    train_df['RE (%)'] = np.zeros_like(final_train_actual)   # 相对误差
    
    # 新增: 添加MAE、MSE、RMSE到训练集数据框
    train_df['MAE'] = np.abs(final_train_actual - final_train_pred)
    train_df['MSE'] = np.square(final_train_actual - final_train_pred)
    train_df['RMSE'] = np.sqrt(train_df['MSE'])
    
    # 避免除以零
    mask_train = final_train_actual != 0
    train_df.loc[mask_train, 'APE (%)'] = np.abs((final_train_actual[mask_train] - final_train_pred[mask_train]) / final_train_actual[mask_train]) * 100
    train_df.loc[mask_train, 'RE (%)'] = ((final_train_pred[mask_train] - final_train_actual[mask_train]) / final_train_actual[mask_train]) * 100
    train_df['Set'] = 'Train'
    
    # 测试集数据
    test_df = pd.DataFrame()
    for i, feature in enumerate(final_features_rfe):
        # 使用反标准化获取原始特征值
        test_df[feature] = final_scaler.inverse_transform(final_test_features)[:, i]
    
    test_df['Actual'] = final_test_actual
    test_df['Predicted'] = final_test_pred
    test_df['Residual'] = test_residuals
    test_df['APE (%)'] = np.zeros_like(final_test_actual)  # 绝对百分比误差
    test_df['RE (%)'] = np.zeros_like(final_test_actual)   # 相对误差
    
    # 新增: 添加MAE、MSE、RMSE到测试集数据框
    test_df['MAE'] = np.abs(final_test_actual - final_test_pred)
    test_df['MSE'] = np.square(final_test_actual - final_test_pred)
    test_df['RMSE'] = np.sqrt(test_df['MSE'])
    
    # 避免除以零
    mask_test = final_test_actual != 0
    test_df.loc[mask_test, 'APE (%)'] = np.abs((final_test_actual[mask_test] - final_test_pred[mask_test]) / final_test_actual[mask_test]) * 100
    test_df.loc[mask_test, 'RE (%)'] = ((final_test_pred[mask_test] - final_test_actual[mask_test]) / final_test_actual[mask_test]) * 100
    test_df['Set'] = 'Test'
    
    # 合并数据
    combined_df = pd.concat([train_df, test_df], ignore_index=True)
    
    # 创建新的组合表格，按要求格式
    # 先确保训练集和测试集有相同数量的行，否则需要填充
    max_rows = max(len(train_df), len(test_df))
    
    # 创建空的DataFrame
    formatted_df = pd.DataFrame({
        '训练集真实值': pd.Series(dtype='float64', index=range(max_rows)),
        '训练集预测值': pd.Series(dtype='float64', index=range(max_rows)),
        '测试集真实值': pd.Series(dtype='float64', index=range(max_rows)),
        '测试集预测值': pd.Series(dtype='float64', index=range(max_rows))
    })
    
    # 填充训练集数据
    formatted_df.loc[:len(train_df)-1, '训练集真实值'] = train_df['Actual'].values
    formatted_df.loc[:len(train_df)-1, '训练集预测值'] = train_df['Predicted'].values
    
    # 填充测试集数据
    formatted_df.loc[:len(test_df)-1, '测试集真实值'] = test_df['Actual'].values
    formatted_df.loc[:len(test_df)-1, '测试集预测值'] = test_df['Predicted'].values
    
    # 全数据库预测结果
    all_data_df = pd.DataFrame()
    
    # 使用原始特征
    for i, feature in enumerate(final_features_rfe):
        all_data_df[feature] = X_final_original[:, i]
    
    all_data_df['真实值'] = y_all_np
    all_data_df['预测值'] = all_predictions
    all_data_df['残差'] = y_all_np - all_predictions
    all_data_df['APE (%)'] = individual_abs_percent_errors  # 绝对百分比误差
    all_data_df['RE (%)'] = individual_rel_errors  # 相对误差
    
    # 新增: 添加MAE、MSE、RMSE到全数据集数据框
    all_data_df['MAE'] = individual_abs_errors
    all_data_df['MSE'] = individual_sq_errors
    all_data_df['RMSE'] = individual_rmse
    
    # 添加性能指标
    train_metrics = pd.DataFrame({
        'Set': ['Train'],
        'R2': [r2_score(final_train_actual, final_train_pred)],
        'RMSE': [np.sqrt(mean_squared_error(final_train_actual, final_train_pred))],
        'MAE': [mean_absolute_error(final_train_actual, final_train_pred)],
        'MSE': [mean_squared_error(final_train_actual, final_train_pred)],  # 新增MSE
        'MAPE (%)': [calculate_mape(final_train_actual, final_train_pred)],
        'MRE (%)': [calculate_mre(final_train_actual, final_train_pred)],
        'MRE+SD (%)': [calculate_mre_sd(final_train_actual, final_train_pred)[2]]
    })
    
    test_metrics = pd.DataFrame({
        'Set': ['Test'],
        'R2': [r2_score(final_test_actual, final_test_pred)],
        'RMSE': [np.sqrt(mean_squared_error(final_test_actual, final_test_pred))],
        'MAE': [mean_absolute_error(final_test_actual, final_test_pred)],
        'MSE': [mean_squared_error(final_test_actual, final_test_pred)],  # 新增MSE
        'MAPE (%)': [calculate_mape(final_test_actual, final_test_pred)],
        'MRE (%)': [calculate_mre(final_test_actual, final_test_pred)],
        'MRE+SD (%)': [calculate_mre_sd(final_test_actual, final_test_pred)[2]]
    })
    
    all_metrics = pd.DataFrame({
        'Set': ['All Data'],
        'R2': [r2_score(y_all_np, all_predictions)],
        'RMSE': [overall_rmse],
        'MAE': [overall_mae],
        'MSE': [overall_mse],  # 新增MSE
        'MAPE (%)': [overall_mape],
        'MRE (%)': [overall_mre],
        'MRE+SD (%)': [overall_mre_plus_sd]
    })
    
    metrics_df = pd.concat([train_metrics, test_metrics, all_metrics], ignore_index=True)
    
    # 保存到Excel的不同工作表
    with pd.ExcelWriter(os.path.join(viz_save_dir, 'prediction_results.xlsx')) as writer:
        formatted_df.to_excel(writer, sheet_name='Combined Train-Test', index=False)
        combined_df.to_excel(writer, sheet_name='Detailed Predictions', index=False)
        all_data_df.to_excel(writer, sheet_name='All Data Predictions', index=False)
        metrics_df.to_excel(writer, sheet_name='Metrics', index=False)
        
        # 额外保存随机种子信息
        seed_df = pd.DataFrame({'Random Seed': [final_best_seed]})
        seed_df.to_excel(writer, sheet_name='Seed', index=False)
    
    # 5. MAPE 分布直方图
    plt.figure(figsize=(10, 6))
    plt.hist(individual_abs_percent_errors[mask], bins=20, alpha=0.7, color='blue')
    plt.axvline(x=overall_mape, color='red', linestyle='--', 
                label=f'Mean MAPE: {overall_mape:.2f}%')
    plt.xlabel('Absolute Percentage Error (%)')
    plt.ylabel('Frequency')
    plt.title('Distribution of MAPE across All Data')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(viz_save_dir, 'mape_distribution.png'), dpi=300)
    plt.close()
    
    # 6. MRE 分布直方图
    plt.figure(figsize=(10, 6))
    plt.hist(individual_rel_errors[mask], bins=20, alpha=0.7, color='blue')
    plt.axvline(x=overall_mre, color='red', linestyle='--',
                label=f'Mean RE: {overall_mre:.2f}%')
    plt.axvline(x=overall_mre + overall_mre_sd, color='green', linestyle='--',
                label=f'MRE+SD: {overall_mre_plus_sd:.2f}%')
    plt.axvline(x=overall_mre - overall_mre_sd, color='green', linestyle='--')
    plt.xlabel('Relative Error (%)')
    plt.ylabel('Frequency')
    plt.title('Distribution of MRE across All Data')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(viz_save_dir, 'mre_distribution.png'), dpi=300)
    plt.close()
    
    # 新增: 7. MAE 分布直方图
    plt.figure(figsize=(10, 6))
    plt.hist(individual_abs_errors, bins=20, alpha=0.7, color='blue')
    plt.axvline(x=overall_mae, color='red', linestyle='--', 
                label=f'Mean MAE: {overall_mae:.2f}')
    plt.xlabel('Absolute Error')
    plt.ylabel('Frequency')
    plt.title('Distribution of MAE across All Data')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(viz_save_dir, 'mae_distribution.png'), dpi=300)
    plt.close()
    
    # 新增: 8. MSE 分布直方图
    plt.figure(figsize=(10, 6))
    plt.hist(individual_sq_errors, bins=20, alpha=0.7, color='blue')
    plt.axvline(x=overall_mse, color='red', linestyle='--',
                label=f'Mean MSE: {overall_mse:.2f}')
    plt.xlabel('Squared Error')
    plt.ylabel('Frequency')
    plt.title('Distribution of MSE across All Data')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(viz_save_dir, 'mse_distribution.png'), dpi=300)
    plt.close()
    
    # 新增: 9. RMSE 分布直方图
    plt.figure(figsize=(10, 6))
    plt.hist(individual_rmse, bins=20, alpha=0.7, color='blue')
    plt.axvline(x=overall_rmse, color='red', linestyle='--',
                label=f'Mean RMSE: {overall_rmse:.2f}')
    plt.xlabel('Root Mean Squared Error')
    plt.ylabel('Frequency')
    plt.title('Distribution of RMSE across All Data')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(viz_save_dir, 'rmse_distribution.png'), dpi=300)
    plt.close()
    
    # 新增: 10. 所有误差指标的箱线图比较
    plt.figure(figsize=(12, 8))
    
    # 注意：我们需要对不同量级的指标进行归一化，否则在一张图上看不清
    # 创建一个新的DataFrame存储需要可视化的指标
    error_metrics_df = pd.DataFrame({
        'MAE': individual_abs_errors / np.mean(individual_abs_errors),
        'RMSE': individual_rmse / np.mean(individual_rmse),
        'MAPE (%)': individual_abs_percent_errors[mask] / np.mean(individual_abs_percent_errors[mask]),
        'RE (%)': np.abs(individual_rel_errors[mask]) / np.mean(np.abs(individual_rel_errors[mask]))
    })
    
    # 创建箱线图
    error_metrics_df.boxplot()
    plt.ylabel('Normalized Value (divided by mean)')
    plt.title('Comparison of Error Metrics (Normalized)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(viz_save_dir, 'error_metrics_boxplot.png'), dpi=300)
    plt.close()

# 打印出最终特征列表
print("\n=== 最终模型使用的6个特征 ===")
for i, feature in enumerate(final_features_rfe):
    print(f"{i+1}. {feature}")
print("\n请确保预测程序计算这些特征并按此顺序排列")

# ====================== 15. 可视化训练集与测试集特征分布（Pair Plot） ======================
import seaborn as sns

# 创建副本，并添加 dataset 标签列
feature_rfe_copy = feature_rfe.copy()
feature_rfe_copy['dataset'] = 'train'  # 默认标记为训练集

# 创建测试集特征的 DataFrame
test_df = pd.DataFrame(final_test_features, columns=final_features_rfe)

# 使用 merge 来标记 test 样本（更稳妥，避免浮点误差带来的匹配失败）
merged = feature_rfe_copy.merge(test_df, on=final_features_rfe, how='left', indicator=True)
feature_rfe_copy['dataset'] = np.where(merged['_merge'] == 'both', 'test', 'train')

# 替换图中显示的列名（只替换需要改名的特征）
rename_dict = {
    'Mean_热膨胀(1/k)': 'M-F6',
    'Mean_比热容J/g/k': 'M-F7',
    'Mean_Pyykko(Triple Bond) (pm)': 'M-C4',
    'Var_E13 Electron affinity': 'V-B2'
}
feature_rfe_copy.rename(columns=rename_dict, inplace=True)

# 设置 Seaborn 样式
sns.set(style="ticks")

# 绘制 Pair Plot 图
pair_plot = sns.pairplot(
    feature_rfe_copy,
    hue='dataset',
    diag_kind='hist',
    plot_kws={'alpha': 0.6}
)

# 设置标题
pair_plot.fig.suptitle("训练集与测试集特征分布对比图（Pair Plot）", y=1.02)

# 保存图像
pairplot_path = os.path.join(viz_save_dir, 'pairplot_train_vs_test.png')
pair_plot.savefig(pairplot_path, dpi=300)
plt.close()

print(f"\n✅ 已保存训练集与测试集特征分布对比图（Pair Plot）：{pairplot_path}")


### PCC 去除冗余特征过程：
| 冗余特征对 | 特征对 PCC 值 | 移除特征 | 移除特征与目标变量 PCC 值 |
| --- | --- | --- | --- |
| Nb 和 Mean_C3 enthalpy vaporization | 0.9575 | Nb | 0.1068 |
| Nb 和 Mean_C5 enthalpy atomization | 0.9549 | Nb | 0.1068 |
| Nb 和 Var_S12 Lattice Constants c | -0.9545 | Nb | 0.1068 |
| Si 和 Δa | 0.9545 | Si | -0.0773 |
| Si 和 Var_S10 Lattice Constants a | 0.9539 | Si | -0.0773 |
| Si 和 Var_S11 Lattice Constants b | 0.9539 | Si | -0.0773 |
| Al 和 ΔTm | 0.9642 | ΔTm | -0.0613 |
| Al 和 Var_C1 temperature melting | 0.9601 | Var_C1 temperature melting | -0.0572 |
| Al 和 Var_C4 enthalpy melting | 0.9724 | Var_C4 enthalpy melting | -0.1219 |
| Al 和 Var_热导率W/(mk) | 0.9583 | Var_热导率W/(mk) | -0.0988 |
| Al 和 Var_热膨胀(1/k) | 0.9987 | Var_热膨胀(1/k) | -0.1315 |
| Hf 和 Var_A1 atomic number | 0.9724 | Hf | -0.1788 |
| Hf 和 Var_A6 atomic weight | 0.9837 | Hf | -0.1788 |
| B 和 Mean_Neutron Cross Section | 0.9743 | B | -0.1307 |
| B 和 Mean_Neutron Mass Absorption | 0.9996 | B | -0.1307 |
| B 和 Mean_电阻率(mΩ) | 1.0000